# Simon's Algorithm – Reference Implementation (s = '011')

**Quantum Computing Assignment 2 — Part A**

This notebook implements Simon's algorithm for the 3-bit secret string **s = '011'**,
following the IBM Quantum Learning module and the Qiskit textbook notebook.

---

## 1  Background

Simon's problem: given a black-box function $f:\{0,1\}^n \to \{0,1\}^n$ that satisfies

$$f(x) = f(x \oplus s)$$

for some hidden string $s$, find $s$.

A classical algorithm needs $O(2^{n/2})$ queries (birthday paradox), while Simon's
quantum algorithm needs only $O(n)$ queries — an **exponential speedup**.

Each circuit run produces a measurement outcome $u$ satisfying

$$u \cdot s = 0 \pmod{2}.$$

Collecting $n-1$ linearly independent such strings determines $s$ uniquely.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
from collections import Counter

print('Qiskit and Aer imported successfully.')

---

## 2  Oracle Construction

### Design principle

We need an oracle $|x\rangle|0\rangle \to |x\rangle|f(x)\rangle$ where
$f(x) = f(x \oplus s)$ for every $x$.  
For $s = 011$ (i.e. $s = (s_0, s_1, s_2) = (0, 1, 1)$), a valid 2-to-1 function is:

$$f(x_0, x_1, x_2) = (x_0,\; 0,\; x_2 \oplus x_1)$$

**Verification that $f$ is 2-to-1 with period $s = (0,1,1)$:**

$$f(x_0, x_1 \oplus 1, x_2 \oplus 1)
  = (x_0,\;0,\;(x_2\oplus 1)\oplus(x_1\oplus 1))
  = (x_0,\;0,\;x_2\oplus x_1) = f(x_0, x_1, x_2). \checkmark$$

### Circuit

Let $k = 1$ be the index of the first `'1'` in $s = 011$.  
The oracle circuit acts on qubits $\{q_0, q_1, q_2\}$ (input) and $\{q_3, q_4, q_5\}$ (output):

| Gate | Effect |
|------|--------|
| CNOT($q_0$, $q_3$) | $\text{out}_0 \leftarrow x_0$ |
| CNOT($q_2$, $q_5$) | $\text{out}_2 \leftarrow x_2$ |
| CNOT($q_1$, $q_5$) | $\text{out}_2 \leftarrow x_2 \oplus x_1$ |

Qubit $q_4$ (output bit 1) stays $|0\rangle$, implementing the constant $0$ in $f$.

In [ ]:
def simon_oracle(s: str) -> QuantumCircuit:
    """
    Build a valid 2-to-1 Simon oracle for secret string s.

    Construction (k = index of first '1' in s):
      output[i] = 0          if i == k
      output[i] = x[i]^x[k] if s[i]=='1' and i != k
      output[i] = x[i]       if s[i]=='0'

    This guarantees f(x) = f(x XOR s) for all x.
    """
    n = len(s)
    qc = QuantumCircuit(2 * n, name='Oracle')

    # Index of the first '1' in s (canonical position)
    k = s.index('1')

    # Copy all bits except the canonical position
    for i in range(n):
        if i != k:
            qc.cx(i, n + i)

    # For every other '1' in s, XOR that output bit with input[k]
    for i in range(n):
        if s[i] == '1' and i != k:
            qc.cx(k, n + i)

    return qc


# Instantiate for s = '011'
s = '011'
oracle = simon_oracle(s)
print(f"Oracle circuit for s = '{s}':")
print(oracle.draw(output='text'))

---

## 3  Full Simon Circuit

The complete circuit has four stages:

1. **1st Hadamard layer** — puts the input register into an equal superposition of all $2^n$ computational basis states.
2. **Oracle** — entangles input and ancilla registers, encoding $f$.
3. **2nd Hadamard layer** — applies quantum interference so that only $u$ satisfying $u \cdot s = 0$ survive.
4. **Measurement** — of the input register only.

In [ ]:
def simon_circuit(s: str) -> QuantumCircuit:
    """Build the complete Simon's algorithm circuit."""
    n = len(s)
    qc = QuantumCircuit(2 * n, n)

    # ── Stage 1: first Hadamard layer ─────────────────────────────────────
    qc.h(range(n))
    qc.barrier(label='H₁')

    # ── Stage 2: oracle ───────────────────────────────────────────────────
    oracle = simon_oracle(s)
    qc.compose(oracle, inplace=True)
    qc.barrier(label='Oracle')

    # ── Stage 3: second Hadamard layer ────────────────────────────────────
    qc.h(range(n))
    qc.barrier(label='H₂')

    # ── Stage 4: measure input register ──────────────────────────────────
    qc.measure(range(n), range(n))
    return qc


qc_3 = simon_circuit(s)
print(f"Full Simon circuit for s = '{s}':")
print(qc_3.draw(output='text'))

---

## 4  Simulation

In [ ]:
SHOTS = 2000
simulator = AerSimulator()

job = simulator.run(transpile(qc_3, simulator), shots=SHOTS)
raw_counts = job.result().get_counts()

# Qiskit returns bit strings with qubit-0 as the RIGHTMOST character.
# Reverse each key so that bit-0 is leftmost (matching our index convention).
counts = {k[::-1]: v for k, v in raw_counts.items()}
counts = dict(sorted(counts.items(), key=lambda x: -x[1]))

print(f"Measurement outcomes for s = '{s}' ({SHOTS} shots):\n")
print(f"  {'Outcome':10s} {'Count':>6s}   u·s (mod 2)")
print("  " + "-" * 35)
for u, cnt in counts.items():
    dot = sum(int(u[i]) * int(s[i]) for i in range(len(s))) % 2
    print(f"  {u:10s} {cnt:6d}   {dot}")

In [ ]:
# Histogram
plot_histogram(counts, title=f"Simon's Algorithm: s = '{s}'")
plt.tight_layout()
plt.show()

---

## 5  Part A — Conceptual Explanation

### 5.1  Role of the First Hadamard Layer

Applying $H^{\otimes n}$ to $|0\rangle^n$ creates an equal superposition over all $2^n$ input strings:

$$|\psi_1\rangle = H^{\otimes n}|0\rangle^n \otimes |0\rangle^n
  = \frac{1}{\sqrt{2^n}} \sum_{x \in \{0,1\}^n} |x\rangle|0\rangle^n.$$

This allows the oracle to be queried on **all** possible inputs simultaneously — the central resource of quantum parallelism.

### 5.2  Role of the Oracle

The oracle is a unitary $U_f$ that acts as:

$$U_f |x\rangle|0\rangle = |x\rangle|f(x)\rangle.$$

After application:

$$|\psi_2\rangle = \frac{1}{\sqrt{2^n}} \sum_x |x\rangle|f(x)\rangle.$$

Because $f(x) = f(x \oplus s)$, the terms naturally group into pairs:

$$|\psi_2\rangle = \frac{1}{\sqrt{2^{n-1}}} \sum_{x_0} \frac{|x_0\rangle + |x_0 \oplus s\rangle}{\sqrt{2}} \otimes |f(x_0)\rangle,$$

where the sum runs over canonical representatives $x_0$ (one per pair).  
Measuring the second register collapses to a specific $f$-value, leaving the first register in the state $\tfrac{1}{\sqrt{2}}(|x_0\rangle + |x_0 \oplus s\rangle)$ for some unknown $x_0$.

### 5.3  Role of the Second Hadamard Layer

Applying $H^{\otimes n}$ to $\tfrac{1}{\sqrt{2}}(|x_0\rangle + |x_0 \oplus s\rangle)$:

$$H^{\otimes n}\bigl(|x_0\rangle + |x_0 \oplus s\rangle\bigr)
  \propto \sum_{u \in \{0,1\}^n} (-1)^{u \cdot x_0}\bigl(1 + (-1)^{u \cdot s}\bigr)|u\rangle.$$

The factor $(1 + (-1)^{u \cdot s})$ equals $2$ when $u \cdot s = 0 \pmod{2}$ and $0$ otherwise.  
Destructive interference **cancels every $|u\rangle$ for which $u \cdot s = 1$**, leaving only
uniform amplitude over the $2^{n-1}$ strings satisfying $u \cdot s = 0 \pmod 2$.

### 5.4  Role of the Measurement

Measuring the first (input) register samples uniformly from the set
$\{u : u \cdot s = 0 \pmod 2\}$.  
Each run of the circuit produces an independent constraint on $s$.  
After $n-1$ linearly independent constraints are collected, classical Gaussian
elimination over $\mathbb{F}_2$ uniquely determines $s$.

### 5.5  Why Do All Outcomes Satisfy $u \cdot s = 0$?

Formally, from the amplitude calculation above, the probability of measuring
any $u$ with $u \cdot s = 1 \pmod 2$ is exactly **zero**.  
Only bit strings orthogonal to $s$ (over $\mathbb{F}_2$) receive non-zero
amplitude after the second Hadamard, so every measurement outcome satisfies
the constraint by quantum-mechanical necessity.

---

## 6  Verification

In [ ]:
n = len(s)
non_trivial = [u for u in counts if u != '0' * n]

print(f"Secret string: s = '{s}'")
print(f"Expected valid outcomes (u·s = 0 mod 2):")
valid = [u for u in [format(i, f'0{n}b') for i in range(2**n)]
         if sum(int(u[j]) * int(s[j]) for j in range(n)) % 2 == 0]
print(f"  {valid}")
print(f"\nObserved outcomes: {list(counts.keys())}")

all_ok = all(
    sum(int(u[j]) * int(s[j]) for j in range(n)) % 2 == 0
    for u in counts
)
print(f"\nAll outcomes satisfy u·s = 0 (mod 2): {all_ok}")